# A·B 교차 추가실험 — Qwen3-8B · 693개

**확인할 것:** 풀이가 있을 때 억제 절의 정답 수정 감소가 완화되는지, 수정 절을 함께 두면 도움이 되는지 봅니다.

기존 초기 오답 **77문항**만 사용합니다. 최초 답변과 풀이, 방어 문구는 기존 그대로입니다.

| 방어 | R1: 정답 선택지만 | R2: 같은 선택지 + 기존 풀이 |
|---|---:|---:|
| S0 방어 없음 | 기존 231개 | 기존 231개 |
| SC 공통 문장만 | 기존 231개 | **새 231개** |
| SI 공통 + 억제 절 | 기존 231개 | **새 231개** |
| SR 공통 + 수정 절 | 기존 231개 | **새 231개** |
| S1 공통 + 두 절 | 기존 231개 | 기존 231개 |

기존 **1,617개** 재사용 + 신규 **693개** = 전체 비교 **2,310개**. 새 생성은 배치16으로 **44배치**입니다.

**A100 선택 → 1번 설치 → 세션 다시 시작 → 2~8번 순서대로 실행. 설정 수정은 필요 없습니다.**

자료 ZIP은 `sycophancy_qwen_8b_ab693_inputs.zip`을 2번 셀에서 한 번 올립니다. 이전 결과 ZIP은 추가로 올리지 않습니다.

정답 유지나 오답 압박에 대한 새 실험은 포함하지 않습니다. 동일 문항의 탐색적 후속 분석이며, 기존/신규 실행 시드가 다릅니다.


## 1. 설치

이 셀만 먼저 실행합니다. 설치가 끝나면 **런타임 → 세션 다시 시작**을 누르고 2번부터 실행하세요.


In [ ]:
import os, sys, subprocess
INSTALL_RAN_IN_THIS_KERNEL = True
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                   "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
packages = ["transformers==4.57.6", "accelerate==1.12.0", "bitsandbytes==0.48.2",
            "huggingface-hub==0.36.0", "tokenizers==0.22.2", "numpy==2.2.6", "pandas==2.2.3",
            "pyarrow==20.0.0", "matplotlib==3.10.3", "scipy==1.15.3", "tqdm==4.67.1"]
print("1/3 패키지 설치 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("2/3 Transformers 설치 파일 확인·복구 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                       "--no-deps", "--no-cache-dir", "transformers==4.57.6"])
print("3/3 별도 Python 프로세스에서 실제 Qwen3 import 검사", flush=True)
PROBE = """
from transformers.generation import GenerationMixin
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from transformers import BitsAndBytesConfig
import torch, accelerate, bitsandbytes, transformers
print("Qwen3 / GenerationMixin / 양자화 import OK")
print("torch:", torch.__version__, "transformers:", transformers.__version__)
"""
result = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("설치/import 검사 실패. 이후 셀로 넘어가지 말고 출력을 공유하세요.")
print("설치 및 import 점검 완료. 지금 세션을 다시 시작하고 2번부터 실행하세요.", flush=True)


## 2. Drive 연결·자료 ZIP

Drive 연결을 허용하고 이번 `inputs.zip` 하나를 선택하세요. ZIP을 직접 풀 필요는 없습니다. 다음 접속부터는 Drive 자료를 자동으로 불러옵니다.


In [ ]:
import os, sys, hashlib, io, zipfile, json, importlib
from pathlib import Path
if globals().get("INSTALL_RAN_IN_THIS_KERNEL", False):
    raise RuntimeError("런타임 → 세션 다시 시작을 누른 뒤 2번부터 실행하세요.")
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                    "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
from google.colab import drive, files
drive.mount("/content/drive")
INPUT_ZIP_NAME = "sycophancy_qwen_8b_ab693_inputs.zip"
INPUT_SHA256 = "c5aa4a435398a9ec1fabdd6f570f6c44cfef0e12e30b8050e0e94f2daf5cc05c"
CACHE_DIR = Path("/content/drive/MyDrive/sycophancy_qwen/inputs") / INPUT_SHA256[:12]
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = CACHE_DIR / INPUT_ZIP_NAME
if ZIP_PATH.exists():
    raw = ZIP_PATH.read_bytes()
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive의 자료 ZIP이 손상됐습니다. 이 오류 출력을 공유하세요.")
    print("Drive에 보관된 자료 ZIP을 불러왔습니다. 다시 업로드할 필요 없습니다.")
else:
    print("아래 파일 선택 버튼에서", INPUT_ZIP_NAME, "하나를 선택하세요.", flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("자료 ZIP 하나만 선택하고 2번을 다시 실행하세요.")
    raw = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("지정한 자료 ZIP과 내용이 다릅니다. 이번 inputs.zip을 선택하고 2번을 다시 실행하세요.")
    temp = ZIP_PATH.with_suffix(".zip.tmp")
    temp.write_bytes(raw)
    if hashlib.sha256(temp.read_bytes()).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive 저장 검사 실패. 저장 공간과 연결을 확인하고 2번을 다시 실행하세요.")
    temp.replace(ZIP_PATH)
    print("자료 ZIP을 Drive에 보관했습니다. 다음부터 자동으로 불러옵니다.")
CODE_DIR = Path("/content") / ("qwen8b_ab693_" + INPUT_SHA256[:12])
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if any(not (CODE_DIR / name).resolve().is_relative_to(CODE_DIR.resolve()) for name in z.namelist()):
        raise RuntimeError("잘못된 ZIP 내부 경로")
    z.extractall(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
for name in ["protocol", "engine", "analysis", "runtime_check", "progress_ui",
             "ablation_protocol", "ab_protocol", "ab_run", "ab_analysis"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from runtime_check import check_runtime
RUNTIME = check_runtime()
from ab_protocol import load_source, default_config, freeze, DEFENSES
from ab_run import ABBackend, bind_backend, run_followups, scan_saved, export_results, write_status
from progress_ui import activity
from protocol import read_json
import pandas as pd
from IPython.display import display, Image
print("자료 준비 완료. 3번부터 실행하세요.")


## 3. 기존 자료 확인·설정 고정

**77문항 / 재사용 1,617개 / 신규 693개 / 44배치**를 확인하세요. 기존 결과와 메시지가 다르면 중단합니다.


In [ ]:
from ab_protocol import load_history, load_manifest, DEFENSE_LABELS
from ab_analysis import score_records
with activity("기존 최초 답변·A/B 원본 및 77문항 대응 검사"):
    SOURCE = load_source(CODE_DIR / "source")
    HISTORY = load_history(SOURCE)
    MANIFEST = load_manifest()
CONFIG = default_config(SOURCE)
STUDY_ROOT = Path("/content/drive/MyDrive/sycophancy_qwen/sycophancy_qwen3_8b_ab693_a100_b16_v1")
with activity("693개 새 작업·원본 재사용·분석 계획 고정"):
    FROZEN, JOBS = freeze(STUDY_ROOT, SOURCE, CONFIG)
print("초기 오답 77문항 | 기존 후속 응답 1,617개 재사용 | 새 응답 693개 | 44배치")
print("새 생성: R2 × SC·SI·SR만. 최초 답변 / R1 / R2 S0·S1 재생성은 0개.")
HISTORICAL_TABLE = score_records(SOURCE, HISTORY["records"]).groupby(["defense","condition"]).final_correct.mean().unstack() * 100
display(HISTORICAL_TABLE.reindex(["S0","SC","SI","SR","S1"]).round(2))
print("빈 R2 셀 세 개를 채웁니다. 각 셀은 77문항 × 3표현 = 231응답입니다.")
print("조건:", DEFENSE_LABELS)
print("설정: A100 / Qwen3-8B / NF4·BF16 / 비사고 / 배치16 / 출력 최대192토큰")
print("기존 A/B의 풀이·최초 답변·방어 문구를 그대로 사용합니다.")
print("Drive 저장:", STUDY_ROOT)
print("동결 해시:", FROZEN["study_hash"])


## 4. 모델 준비

모델을 다운로드하고 원본 A100·핵심 패키지·생성 설정과 비교합니다. 입력 693개의 길이도 점검합니다. A100 환경이 다르면 출력된 차이를 공유하세요.


In [ ]:
with activity("저장 배치 확인"):
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if len(SAVED) == 44:
    BACKEND = None
    print("이미 693개가 저장돼 모델 로딩을 건너뜁니다. 7번에서 분석할 수 있습니다.")
else:
    if globals().get("BACKEND") is None:
        with activity("Qwen3-8B 가중치 다운로드·A100 적재"):
            BACKEND = ABBackend(SOURCE, CONFIG)
    bind_backend(STUDY_ROOT, FROZEN, SOURCE, BACKEND, JOBS)
    print("GPU:", BACKEND.environment["gpu"])
    print("원본과 모델/생성 설정 일치. NF4 / BF16 / 비사고 / 배치 16 / 최대 192토큰")
    if BACKEND.source_environment_notes:
        print("기록한 기타 환경 차이:", BACKEND.source_environment_notes)
    print("모델 및 전체 입력 점검 완료. 5번으로 진행하세요.")


## 5. 첫 배치 점검

첫 16개를 생성해 Drive 저장과 출력 형식을 확인합니다. 이 16개도 본 실험에 포함하며 다시 생성하지 않습니다.


In [ ]:
SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if 0 not in SAVED:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND, max_new_batches=1)
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
else:
    print("첫 배치는 이미 저장되어 재사용합니다.")
from ab_analysis import score_records
FIRST_CHECK = score_records(SOURCE, SAVED[0]["records"])
print("첫 배치 Drive 저장·해시 검사 통과: 16/16개")
print("형식 실패:", int(FIRST_CHECK.invalid.sum()), "/16 | 잘림:", int(FIRST_CHECK.truncated.sum()), "/16")
print("현재 전체 저장:", sum(len(x["records"]) for x in SAVED.values()), "/693")
if FIRST_CHECK.invalid.all():
    raise RuntimeError("첫 배치 16개가 모두 형식 실패입니다. 저장 자료는 보존하고 이 출력을 공유하세요.")
print("6번을 실행하면 저장된 배치를 건너뛰고 이어갑니다.")


## 6. 남은 응답 생성

진행률·저장 개수·예상 남은 시간이 표시됩니다. **693/693, 44/44, complete=true**를 확인하세요. 배치마다 자동 저장합니다.


In [ ]:
if BACKEND is None:
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
    if len(SAVED) != 44:
        raise RuntimeError("4번 모델 준비부터 다시 실행하세요.")
    STATUS = write_status(STUDY_ROOT, SAVED, JOBS, CONFIG, "complete")
    print("이미 모든 배치가 저장되어 재생성하지 않습니다.")
else:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND)
    if STATUS["complete"]:
        BACKEND.close()
        BACKEND = None
print(json.dumps(STATUS, ensure_ascii=False, indent=2))


## 7. 자동 분석

문항별로 표현 3개를 묶어 비교합니다. 두 핵심 비교는 97.5% 구간을 사용합니다. 77문항을 231개의 독립 표본으로 계산하지 않습니다.


In [ ]:
from ab_analysis import analyze
with activity("공통 77문항 자동 채점·두 핵심 비교·그래프"):
    REPORT_DIR, TABLE, EFFECTS, ASSESSMENT = analyze(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS)
print("정답 수정률 (%): 같은 초기 오답 77문항, 각 셀 231응답")
display(TABLE.pivot(index="defense",columns="condition",values="final_correct_percent").reindex(["S0","SC","SI","SR","S1"]).round(2))
print("잘못된 답 유지·다른 오답·형식 실패까지 포함한 상세표")
display(TABLE.rename(columns={"condition":"피드백", "defense":"방어", "n_items":"문항수",
    "n_responses":"응답수", "correct_count":"정답 수정 수", "final_correct_percent":"정답 수정률",
    "initial_wrong_retained_percent":"초기 오답 유지율", "other_wrong_change_percent":"다른 오답",
    "invalid_percent":"형식 실패", "truncated_percent":"잘림"}).round(2))
print("핵심 비교: 문항 단위 97.5% 구간, 두 비교 보정. 양수의 의미는 아래 설명을 확인하세요.")
display(EFFECTS[EFFECTS.analysis_role=="primary"][["label_ko","n_items","difference_pp","ci_low_pp","ci_high_pp"]].round(2))
for message in ASSESSMENT["messages_ko"]:
    print(message)
print(ASSESSMENT["scope"])
print("풀이 내용과 길이의 효과를 구분한 실험은 아닙니다. 기존 응답과 새 응답은 서로 다른 시드·실행 시점의 결과입니다.")
print("보조 절 비교: clause_evidence_effects.csv / 문항별 비교: item_contrasts.csv")
for name in ["01_correction_comparison.png","02_primary_contrasts.png"]:
    display(Image(filename=str(REPORT_DIR/name)))
print("분석 저장:", REPORT_DIR)


## 8. 결과 ZIP 다운로드

결과 ZIP은 Drive에도 저장됩니다. 중간에 중단했을 때도 저장된 자료를 내보낼 수 있습니다. 최종 분석은 693개 완료 후 가능합니다.


In [ ]:
with activity("결과 ZIP을 Drive에 저장"):
    RESULT_ZIP = export_results(STUDY_ROOT)
print("완료 상태:", read_json(STUDY_ROOT / "status.json") if (STUDY_ROOT / "status.json").exists() else "미실행")
files.download(str(RESULT_ZIP))


## 저장·재개

저장: `내 드라이브/sycophancy_qwen/sycophancy_qwen3_8b_ab693_a100_b16_v1/`

런타임이 종료되어 새 A100을 받으면 **1번 설치 → 세션 다시 시작 → 2~8번**을 실행합니다. 저장된 배치는 건너뛰며, 미저장 배치만 재개합니다. 기존 셀이 아직 실행 중이면 중복 실행하지 마세요.

제작 환경에서는 가상 응답으로 저장·재개·채점·통계를 검증했습니다. 실제 A100 추론은 이 노트북에서 실행합니다.
